# Vertical background-flow shear of topographic-dominant eddies

This compact notebook asks whether topographic-dominant eddies—especially AEs that appear to tilt downstream on the shelf—experience vertical background-flow shear consistent with their tilt. It uses the same 2:1 topographic-dominance, depth ≤3000 m, and tilt ≥5 km definition as `plan_topo_dom_eddies/topographic_dominated_eddies.ipynb`.

The primary shear is the thickness-weighted **0–200 m mean minus 200–500 m mean** from the centred 91-day climatology. It is not velocity at exactly 200 and 500 m. Every qualifying eddy-day is retained; uncertainty resamples whole eddies.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

HERE = Path.cwd().resolve()
ANALYSIS_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS_ROOT is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or one of its subfolders.')
FLOW_ROOT = ANALYSIS_ROOT / 'beta_effect_background_flow'
CASE_ROOT = ANALYSIS_ROOT / 'case_studies'
for path in (ANALYSIS_ROOT, FLOW_ROOT, CASE_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import seacofs_tilt_tools as tilt
from background_flow_tools import BackgroundConfig, load_background_cache
from case_study_tools import PVAlignmentConfig, add_pv_alignment_diagnostics

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

## 1. Load the topographic-dominant population and background cache

In [ ]:
DOMINANCE_FACTOR = 2.0
MAX_DEPTH_m = 3000.0
MIN_TILT_km = 5.0
SHELF_LON = 154.75
RELATION_SAMPLE = 'on_shelf'  # use 'all' to include every topographic-dominant observation
FAMILY = 'clim'
N_BOOT = 2000
RANDOM_SEED = 42

paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
eddies, _ = tilt.load_tilt_tables(paths)
eddies = tilt.add_region_labels(eddies, grid)
eddies = tilt.add_pv_gradient_terms(
    eddies, grid, core_mean=True, frac=1.0, surface_method='esp_gaussian',
    averaging='nonlinear', use_max_abs_w=True, max_depth_m=1000, use_cache=True)
config = PVAlignmentConfig(dominance_factor=DOMINANCE_FACTOR,
                           max_topo_depth_m=MAX_DEPTH_m,
                           min_tilt_distance_km=MIN_TILT_km)
d = add_pv_alignment_diagnostics(eddies, config)
topographic = d[(d.topo_plan_ratio_smooth >= np.log(DOMINANCE_FACTOR))
                & (d.TiltDis >= MIN_TILT_km)
                & (d.h <= MAX_DEPTH_m)
                & d.direction_valid].copy()

background = load_background_cache(BackgroundConfig())
background = background.drop(columns=['ic', 'jc'], errors='ignore')
topographic = topographic.merge(background, on=['Eddy', 'Day'], how='inner', validate='one_to_one')
topographic = topographic.sort_values(['Eddy', 'Day']).copy()
theta = np.deg2rad(topographic.TiltDir.astype(float))
topographic['tilt_east_km'] = topographic.TiltDis * np.sin(theta)
topographic['tilt_north_km'] = topographic.TiltDis * np.cos(theta)
topographic['shelf_class'] = np.where(topographic.lon < SHELF_LON, 'on_shelf', 'off_shelf')
display(topographic.groupby('Cyc').agg(observations=('Eddy','size'),
    eddies=('Eddy','nunique'), median_depth_m=('h','median'),
    median_tilt_km=('TiltDis','median')).round(2))

## 2. Construct the 200–500 m layer and vertical shear

In [ ]:
for component in ('east', 'north'):
    surface = f'{FAMILY}_surface_{component}_ms'
    upper = f'{FAMILY}_200_{component}_ms'       # thickness-weighted 0–200 m mean
    full500 = f'{FAMILY}_500_{component}_ms'    # thickness-weighted 0–500 m mean
    deep = f'{FAMILY}_200_500_{component}_ms'
    topographic[deep] = (500 * topographic[full500] - 200 * topographic[upper]) / 300
    topographic[f'{FAMILY}_upper_minus_deep_{component}_ms'] = topographic[upper] - topographic[deep]

depth_labels = {'surface': 'Surface', '200': '0–200 m', '200_500': '200–500 m'}
colors = {'AE':'firebrick', 'CE':'royalblue'}

## 3. Flow profile and maps of subsurface shear

In [ ]:
def cluster_bootstrap_median(frame, column, cluster='Eddy', n_boot=N_BOOT, seed=RANDOM_SEED):
    q = frame[[cluster, column]].dropna()
    clusters = {key: part[column].to_numpy(float) for key, part in q.groupby(cluster)}
    keys = np.asarray(list(clusters))
    if not len(keys): return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot)
    for b in range(n_boot):
        sampled = rng.choice(keys, len(keys), replace=True)
        boot[b] = np.median(np.concatenate([clusters[key] for key in sampled]))
    return np.median(q[column]), *np.nanpercentile(boot, [2.5, 97.5])

rows = []
for component in ('east','north'):
    for depth in depth_labels:
        col = f'{FAMILY}_{depth}_{component}_ms'
        for cyc, part in topographic.groupby('Cyc'):
            med, lo, hi = cluster_bootstrap_median(part, col, seed=RANDOM_SEED+len(rows))
            rows.append(dict(family=FAMILY, component=component, depth=depth, Cyc=cyc,
                observations=part[col].notna().sum(), eddies=part.loc[part[col].notna(),'Eddy'].nunique(),
                median=med, ci_low=lo, ci_high=hi))
flow_summary = pd.DataFrame(rows)
display(flow_summary.round(4))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 5), constrained_layout=True)
family = FAMILY

ax = axes[0]
for comp, fmt in {'east':'o-', 'north':'s--'}.items():
    p = flow_summary.query('family == @family and component == @comp')
    for off, cyc in zip((-.08,.08), ('AE','CE')):
        q = p[p.Cyc.eq(cyc)].set_index('depth').reindex(depth_labels)
        m = q['median']
        ax.errorbar(np.arange(3)+off, m, [m-q.ci_low, q.ci_high-m],
                    fmt=fmt, capsize=3, color=colors[cyc],
                    label=f"{cyc} {'zonal' if comp=='east' else 'meridional'}")
ax.axhline(0, color='k', lw=.8)
ax.set(xticks=range(3), ylabel='Background flow (m s$^{-1}$)')
ax.set_xticklabels(depth_labels.values(), rotation=15)
ax.legend(fontsize=8)

panels = [('AE','east'), ('CE','east'), ('AE','north'), ('CE','north')]
cols = [f'{family}_upper_minus_deep_{c}_ms' for _, c in panels]
lim = np.nanpercentile(np.abs(topographic[cols].to_numpy()), 98)

for k, (ax, (cyc, comp)) in enumerate(zip(axes[1:], panels)):
    col = f'{family}_upper_minus_deep_{comp}_ms'
    q = topographic[topographic.Cyc.eq(cyc)].sort_values(col, key=abs)
    sc = ax.scatter(q.xc, q.yc, c=q[col], s=9, alpha=.7,
                    cmap='PiYG', vmin=-lim, vmax=lim)
    ax.contour(grid.X_grid, grid.Y_grid, grid.h, [MAX_DEPTH_m], colors='k', linewidths=.7)
    tilt.lat_lon_contours(ax, grid, land_mask=True, land_clr=colors[cyc])
    ax.text(30, 830, cyc, fontsize=20, weight='bold', color='w')
    ax.set(xlabel='x (km)', ylabel='y (km)' if k==0 else '',
           title=f"{cyc} {'zonal' if comp=='east' else 'meridional'} shear", aspect='equal')
    if k: ax.set_yticklabels([])

fig.colorbar(sc, ax=axes[1:],
    label='0–200 m − 200–500 m background flow (m s$^{-1}$)', shrink=.7)
plt.show()

## 4. Does shear magnitude relate to downstream or cross-shear tilt?

The tilt vector is projected onto the local shear direction. `tilt_parallel_km > 0` means the shallow centre tilts downstream in the direction of the 0–200 minus 200–500 m flow; negative means upstream. `tilt_perpendicular_km` is the signed cross-shear component, and its absolute value measures departure from the shear axis. A downstream-shear explanation for shelf AEs predicts a positive parallel component and, potentially, a positive relationship between shear magnitude and parallel tilt.

In [ ]:
relation_data = (topographic[topographic.shelf_class.eq('on_shelf')].copy()
                 if RELATION_SAMPLE == 'on_shelf' else topographic.copy())
print(f'Shear–tilt relationship sample: {RELATION_SAMPLE}; observations={len(relation_data)}, eddies={relation_data.Eddy.nunique()}')
shear_e = f'{FAMILY}_upper_minus_deep_east_ms'
shear_n = f'{FAMILY}_upper_minus_deep_north_ms'
relation_data['shear_magnitude_ms'] = np.hypot(relation_data[shear_e], relation_data[shear_n])
valid_mag = relation_data.shear_magnitude_ms.where(relation_data.shear_magnitude_ms > 0)
relation_data['tilt_parallel_km'] = (relation_data.tilt_east_km*relation_data[shear_e]
    + relation_data.tilt_north_km*relation_data[shear_n]) / valid_mag
relation_data['tilt_perpendicular_km'] = (-relation_data.tilt_east_km*relation_data[shear_n]
    + relation_data.tilt_north_km*relation_data[shear_e]) / valid_mag
relation_data['tilt_cross_magnitude_km'] = relation_data.tilt_perpendicular_km.abs()

def cluster_bootstrap_correlation(frame, x, y, cluster='Eddy', n_boot=N_BOOT, seed=RANDOM_SEED):
    q = frame[[cluster,x,y]].dropna()
    clusters = {key: part[[x,y]].to_numpy(float) for key, part in q.groupby(cluster)}
    keys = np.asarray(list(clusters))
    if len(keys) < 3: return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed); boot = np.empty(n_boot)
    for b in range(n_boot):
        sampled = rng.choice(keys, len(keys), replace=True)
        sample = np.concatenate([clusters[key] for key in sampled])
        boot[b] = spearmanr(sample[:,0], sample[:,1]).statistic
    observed = spearmanr(q[x],q[y]).statistic
    return observed, *np.nanpercentile(boot,[2.5,97.5])

responses = [('Tilt magnitude','TiltDis'), ('Downstream tilt','tilt_parallel_km'),
             ('Cross-shear magnitude','tilt_cross_magnitude_km')]
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
test_rows = []
for row, cyc in enumerate(('AE','CE')):
    for col, (label, y) in enumerate(responses):
        ax = axes[row,col]
        q = relation_data.loc[relation_data.Cyc.eq(cyc), ['Eddy','shear_magnitude_ms',y]].dropna().copy()
        xlo, xhi = q.shear_magnitude_ms.quantile([.01,.99])
        shown = q[q.shear_magnitude_ms.between(xlo,xhi)]
        ax.hexbin(shown.shear_magnitude_ms, shown[y], gridsize=18, mincnt=1, bins='log',
                  cmap='Reds' if cyc=='AE' else 'Blues', alpha=.75)
        q['shear_bin'] = pd.qcut(q.shear_magnitude_ms, 5, duplicates='drop')
        binned=[]
        for _, part in q.groupby('shear_bin', observed=True):
            med, lo, hi = cluster_bootstrap_median(part,y,n_boot=min(N_BOOT,1000),
                                                    seed=RANDOM_SEED+len(binned))
            binned.append((part.shear_magnitude_ms.median(),med,lo,hi))
        binned=np.asarray(binned)
        ax.errorbar(binned[:,0],binned[:,1],
                    yerr=[binned[:,1]-binned[:,2],binned[:,3]-binned[:,1]],
                    fmt='o-',color='k',capsize=3)
        rho, lo, hi = cluster_bootstrap_correlation(q,'shear_magnitude_ms',y,
                                                     seed=RANDOM_SEED+10*row+col)
        test_rows.append(dict(Cyc=cyc,response=label,observations=len(q),eddies=q.Eddy.nunique(),
                              spearman_rho=rho,ci_low=lo,ci_high=hi))
        ax.axhline(0,color='k',lw=.7); ax.set_xlim(xlo,xhi)
        ax.set(xlabel='0–200 minus 200–500 m shear magnitude (m s$^{-1}$)',
               ylabel=f'{label} (km)', title=f'{cyc}: ρ={rho:.2f} [{lo:.2f}, {hi:.2f}]')
plt.show()
display(pd.DataFrame(test_rows).round(4))

## Interpretation

For the downstream-shear hypothesis, focus first on AEs: (1) is their median `tilt_parallel_km` positive, and (2) does stronger shear correspond to more positive downstream tilt? A weak correlation does not necessarily reject the mechanism when the available shear range is narrow, but consistently positive downstream projection is only directional coexistence—not causal proof. The CE row is a useful polarity comparison. The cross-shear panel shows how much tilt remains unexplained by the shear direction.